In [33]:
#Importing the required libraries
import os
import requests
from pathlib import Path

In [34]:
# Create a session to persist cookies across requests
session = requests.Session()

# Common headers
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0 "),
    "Referer": "https://www.nseindia.com/all-reports",
    "Accept": "application/json"
}

# Visit homepage once to generate cookies
response = session.get("https://www.nseindia.com",headers=HEADERS,timeout=30)

print("Status :", response.status_code)

Status : 200


In [35]:
# Define the base directory for saving downloaded reports
BASE_DIR = Path(r"C:\Users\JAY LODHA\web-scraping-project\data\raw\nse_reports")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print(BASE_DIR)

C:\Users\JAY LODHA\web-scraping-project\data\raw\nse_reports


In [36]:
# Function to get JSON data from a URL
def get_json(url):
    response = session.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()

In [37]:
# Define the API endpoints for daily and monthly reports
API_ENDPOINTS = {
    # Daily APIs
    "CM_DAILY": "https://www.nseindia.com/api/daily-reports?key=CM",
    "INDEX_DAILY": "https://www.nseindia.com/api/daily-reports?key=INDEX",
    "SLBS_DAILY": "https://www.nseindia.com/api/daily-reports?key=SLBS",
    "FO_DAILY": "https://www.nseindia.com/api/daily-reports?key=FO",
    "CD_DAILY": "https://www.nseindia.com/api/daily-reports?key=CD",
    "COM_DAILY": "https://www.nseindia.com/api/daily-reports?key=COM",
    "NBF_DAILY": "https://www.nseindia.com/api/daily-reports?key=NBF",
    "WDM_DAILY": "https://www.nseindia.com/api/daily-reports?key=WDM",
    "CBM_DAILY": "https://www.nseindia.com/api/daily-reports?key=CBM",
    "TRIPARTY_DAILY": "https://www.nseindia.com/api/daily-reports?key=TRI-PARTY",
    "EGR_DAILY": "https://www.nseindia.com/api/daily-reports?key=EGR",
    # Monthly APIs
    "CM_MONTHLY": "https://www.nseindia.com/api/monthly-reports?key=CM",
    "INDEX_MONTHLY": "https://www.nseindia.com/api/monthly-reports?key=INDICES",
    "SLBS_MONTHLY": "https://www.nseindia.com/api/monthly-reports?key=SLBS",
    "FO_MONTHLY": "https://www.nseindia.com/api/monthly-reports?key=FO",
    "CD_MONTHLY": "https://www.nseindia.com/api/monthly-reports?key=CD",
    "COM_MONTHLY": "https://www.nseindia.com/api/monthly-reports?key=COM",
    "IRD_MONTHLY": "https://www.nseindia.com/api/monthly-reports?key=IRD",
    "WDM_MONTHLY": "https://www.nseindia.com/api/monthly-reports?key=WDM",
    "CBM_MONTHLY": "https://www.nseindia.com/api/monthly-reports?key=CBM",
}

In [38]:
# Function to download reports based on the report type (daily or monthly)
def download_reports(api_url, save_folder, report_type):

    # Create folder if it doesn't exist
    save_folder = Path(save_folder)
    save_folder.mkdir(parents=True, exist_ok=True)

    # Fetch JSON
    data = get_json(api_url)

    # DAILY REPORTS
    if report_type.lower() == "daily":

        for section in ["CurrentDay", "PreviousDay", "FutureDay"]:

            reports = data.get(section)

            print(f"\n{section} : {len(reports)} reports")

            for report in reports:

                file_url = report["filePath"] + report["fileActlName"]
                file_name = report["fileActlName"]

                try:
                    response = session.get(file_url,headers=HEADERS,timeout=30)
                    if response.status_code == 200:
                        with open(save_folder / file_name, "wb") as f:
                            f.write(response.content)
                        print("Downloaded :", file_name)

                    else:
                        print("Failed :", response.status_code, file_name)

                except Exception as e:
                    print("Error :", e)

    # MONTHLY REPORTS
    elif report_type.lower() == "monthly":
        print(f"\nMonthly Reports : {len(data)}")
        for report in data:

            file_url = report["filePath"] + report["fileActlName"]
            file_name = report["fileActlName"]

            try:
                response = session.get(file_url,headers=HEADERS,timeout=30)
                if response.status_code == 200:
                    with open(save_folder / file_name, "wb") as f:
                        f.write(response.content)
                    print("Downloaded :", file_name)

                else:
                    print("Failed :", response.status_code, file_name)

            except Exception as e:
                print("Error :", e)

    else:
        print("Invalid report type.")

In [39]:
FOLDER_MAP = {
    "CM": "Equities & SME",
    "INDEX": "Indices",
    "SLBS": "Securities Lending & Borrowing",
    "FO": "Equity Derivatives",
    "CD": "Currency Derivatives",
    "COM": "Commodity Derivatives",
    "IRD": "Interest Rate Derivatives",
    "WDM": "Debt segment",
    "CBM": "Corporate Bond",
    "TRI-PARTY": "Tri-Party",
    "EGR": "EGR",
    "NBF": "NBF"
}

In [40]:
# Download all Daily and Monthly reports

for key, folder_name in FOLDER_MAP.items():
    print(f"{folder_name}")

    # Daily Reports
    daily_api = API_ENDPOINTS.get(f"{key}_DAILY")

    if daily_api:
        print("Downloading Daily Reports...")
        download_reports(api_url=daily_api,save_folder=BASE_DIR/folder_name/"Daily",report_type="daily")
    else:
        print("Daily API not available")

    # Monthly Reports
    monthly_api = API_ENDPOINTS.get(f"{key}_MONTHLY")

    if monthly_api:
        print("Downloading Monthly Reports...")
        download_reports(api_url=monthly_api,save_folder=BASE_DIR/folder_name/"Monthly",report_type="monthly")
    else:
        print("Monthly API not available")

    print()

Equities & SME

CurrentDay : 19 reports
Downloaded : C_VAR1_03072026_1.DAT
Downloaded : C_VAR1_03072026_2.DAT
Downloaded : C_VAR1_03072026_3.DAT
Downloaded : C_VAR1_03072026_4.DAT
Downloaded : C_VAR1_03072026_5.DAT
Downloaded : APPSEC_COLLVAL_03072026.csv
Downloaded : MF_VAR_03072026.csv
Downloaded : series_change.csv
Downloaded : AUB_2026122_03072026.csv
Downloaded : shortselling_03072026.csv
Downloaded : eq_band_changes_03072026.csv
Downloaded : C_STT_03072026.csv
Downloaded : C_STT_IND_03072026.csv
Downloaded : Daily_Settlement_Statistics_03072026.doc
Downloaded : CM_52_wk_High_low_03072026.csv
Downloaded : CM_Latency_stats02072026.csv
Downloaded : FCM_INTRM_BC03072026.DAT
Downloaded : PE_030726.csv
Downloaded : sme_bands_complete_03072026.csv

PreviousDay : 38 reports
Downloaded : C_VAR1_02072026_1.DAT
Downloaded : C_VAR1_02072026_2.DAT
Downloaded : C_VAR1_02072026_3.DAT
Downloaded : C_VAR1_02072026_4.DAT
Downloaded : C_VAR1_02072026_5.DAT
Downloaded : C_VAR1_02072026_6.DAT
Downloa

In [41]:
# Merged APIs
MERGED_APIS = {
    "Capital Market": "https://www.nseindia.com/api/merged-daily-reports?key=favCapital",
    "Derivatives": "https://www.nseindia.com/api/merged-daily-reports?key=favDerivatives",
    "Debt": "https://www.nseindia.com/api/merged-daily-reports?key=favDebt"
}

In [42]:
from urllib.parse import urlparse

def download_merged_reports(api_url, save_folder):

    save_folder.mkdir(parents=True, exist_ok=True)

    data = get_json(api_url)

    print(f"Found {len(data)} reports")

    for report in data:

        download_url = report["link"]

        file_name = Path(urlparse(download_url).path).name

        try:
            response = session.get(download_url, headers=HEADERS, timeout=30)
            response.raise_for_status()

            with open(save_folder/file_name,"wb") as f:
                f.write(response.content)

            print("Downloaded :", file_name)

        except Exception as e:
            print("Failed :", file_name)
            print(e)

In [43]:
for section, api in MERGED_APIS.items():
    print(section)
    download_merged_reports(api,BASE_DIR/"Favorites"/section)

Capital Market
Found 9 reports
Downloaded : BhavCopy_NSE_CM_0_0_0_20260702_F_0000.csv.zip
Downloaded : MA020726.csv
Downloaded : MTO_02072026.DAT
Downloaded : AUB_2026122_03072026.csv
Failed : cat_turnover_020726.xls
404 Client Error: Not Found for url: https://nsearchives.nseindia.com/archives/equities/cat/cat_turnover_020726.xls
Downloaded : eq_band_changes_03072026.csv
Downloaded : top10nifty50_020726.csv
Downloaded : CMVOLT_02072026.CSV
Downloaded : SLB_ELG_SEC_02072026.csv
Derivatives
Found 9 reports
Downloaded : BhavCopy_NSE_FO_0_0_0_20260702_F_0000.csv.zip
Downloaded : BhavCopy_NSE_CD_0_0_0_20260702_F_0000.csv.zip
Downloaded : BhavCopy_NSE_CO_0_0_0_20260702_F_0000.csv.zip
Downloaded : IRF_Bhavcopy020726.zip
Downloaded : FOSett_prce_02072026.csv
Downloaded : fo_secban_03072026.csv
Downloaded : CDSett_prce_02072026.csv
Downloaded : APPSEC_COLLVAL_03072026.csv
Downloaded : MF_VAR_03072026.csv
Debt
Found 3 reports
Downloaded : wdmlist_02072026.csv
Downloaded : cbm_trd20260702.csv
Do

In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
driver = webdriver.Edge()

driver.get("https://www.nseindia.com/all-reports-derivatives")

Archive_tab=driver.find_element(By.CSS_SELECTOR,"a.nav-item.nav-link")
time.sleep(1)
Archive_tab.click()
